[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/49_reasoning_trace_solution.ipynb)

#  Solution: Reasoning Trace Processor

Reference: DeepSeek-R1 (reasoning traces), ReST-EM, and GRPO training where reasoning tokens are masked from KL penalty.

```text
1. Find all <think>/</think> open/close tag pairs
2. Mark tokens between them as reasoning (exclude the tags)
3. Find all <action>/</action> open/close tag pairs
4. Mark tokens between them as action (exclude the tags)
5. Handle multiple sections
```

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
#  SOLUTION

def split_reasoning_action(token_ids, special_map):
    """Separate reasoning and action tokens.

    Used in DeepSeek-R1 style training where:
    - Reasoning (CoT) tokens are treated as context, not directly optimized by RL
    - Action tokens are where the policy gradient applies
    - GRPO applies group-relative advantages only to action tokens
    Reference: DeepSeekMath (Shao et al.), DeepSeek-R1 (Guo et al. 2025).
    """
    L = len(token_ids)
    reason_mask = torch.zeros(L, dtype=torch.bool)
    action_mask = torch.zeros(L, dtype=torch.bool)

    tid_think_open = special_map["<think>"]
    tid_think_close = special_map["</think>"]
    tid_action_open = special_map["<action>"]
    tid_action_close = special_map["</action>"]

    def mark_sections(open_id, close_id, mask):
        i = 0
        while i < L:
            if token_ids[i] == open_id:
                j = i + 1
                while j < L and token_ids[j] != close_id:
                    j += 1
                if j < L:
                    mask[i+1:j] = True
                i = j
            i += 1

    mark_sections(tid_think_open, tid_think_close, reason_mask)
    mark_sections(tid_action_open, tid_action_close, action_mask)

    return reason_mask, action_mask

In [ ]:
#  Verify
ids = torch.tensor([1, 5, 2, 3, 7, 4, 1, 9, 2, 3, 8, 4])
sp = {"<think>": 1, "</think>": 2, "<action>": 3, "</action>": 4}
r, a = split_reasoning_action(ids, sp)
print(f"Reasoning: {r}")
print(f"Action:    {a}")

In [ ]:
from torch_judge import check
check("reasoning_trace")